In [ ]:
# Executing this cell will:

# Disable all TQDM outputs in stdout.
import os

os.environ["DISABLE_TQDM"] = "True"

# Setup the python logger for the Public API
from osekit import setup_logging

setup_logging()  # Overwrites the default logger to

# Accessing audio data between arbitrary timestamps with the Public API [^download]

[^download]: This notebook can be downloaded as **{nb-download}`example_from_events_public.ipynb`**.

The `_static/detections/aplose_results.csv` file contains a `start_datetime` and a `end_datetime` column that represent the start and stop of time regions of interest within the audio dataset.

Our goal here is to plot a spectrogram for each of these time region using `OSEkit`'s `Public API`.

As always in the **Public API**, the first step is to **build the project**.

An `Instrument` can be provided to the `Project` for the WAV data to be converted in pressure units. This will lead the resulting spectra to be expressed in dB SPL (rather than in dB FS):

In [ ]:
from pathlib import Path

audio_folder = Path(r"_static/sample_audio/timestamped")

from osekit.public.project import Project
from osekit.core.instrument import Instrument

project = Project(
    folder=audio_folder,
    strptime_format="%y%m%d_%H%M%S",
    instrument=Instrument(end_to_end_db=150.0),
    timezone="UTC",
)

project.build()

The **Public API** `Project` is now analyzed and organized:

In [ ]:
print(f"{' DATASET ':#^60}")
print(f"{'Begin:':<30}{str(project.origin_dataset.begin):>30}")
print(f"{'End:':<30}{str(project.origin_dataset.end):>30}")
print(f"{'Sample rate:':<30}{str(project.origin_dataset.sample_rate):>30}\n")

print(f"{' ORIGINAL FILES ':#^60}")
import pandas as pd

pd.DataFrame(
    [
        {
            "Name": f.path.name,
            "Begin": f.begin,
            "End": f.end,
            "Sample Rate": f.sample_rate,
        }
        for f in project.origin_files
    ],
).set_index("Name")

Before getting in the audio data, we will parse each time region as an `OSEkit` `Event`:

In [ ]:
from pathlib import Path
import pandas as pd
from osekit.core.event import Event

timestamps_file = Path(r"_static/detections/aplose_results.csv")

# Read the CSV and parse the timestamps as pandas Timestamps
timestamps_df = pd.read_csv(timestamps_file)
timestamps_df["start_datetime"] = pd.to_datetime(timestamps_df["start_datetime"])
timestamps_df["end_datetime"] = pd.to_datetime(timestamps_df["end_datetime"])

# Create the list of events:
events = [
    Event(begin=row.start_datetime, end=row.end_datetime)
    for row in timestamps_df.itertuples()
]

We can even do a bit of filtering: we will
 - keep only the events that are longer than `5s`
 - remove duplicated events (thanks to the `Event.key` property that is a hash of both the begin and end timestamp):

In [ ]:
from pandas import Timedelta

longer_events = [e for e in events if e.duration > Timedelta(seconds=5)]
unique_event_keys = set()
unique_events = []

for event in longer_events:
    if event.key in unique_event_keys:
        continue
    unique_event_keys.add(event.key)
    unique_events.append(event)

Since we will run a spectral transform (we want to plot spectrograms for each event), we need to define the FFT parameters:

In [ ]:
from scipy.signal import ShortTimeFFT
from scipy.signal.windows import hamming

sample_rate = 24_000

sft = ShortTimeFFT(
    win=hamming(512),
    hop=64,  # This will be forced to len(win) if we compute a LTAS
    fs=sample_rate,
)

To **run transforms** in the **Public API**, use the `Transform` class:

In [ ]:
from osekit.utils.audio import Normalization
from osekit.public.transform import Transform, OutputType

transform = Transform(
    output_type=OutputType.SPECTROGRAM,
    sample_rate=sample_rate,
    normalization=Normalization.DC_REJECT,  # Removes the DC component
    fft=sft,
    v_lim=(0.0, 150.0),  # Boundaries of the spectrograms
    colormap="viridis",  # Default value
    name="events_spectrograms",
)

We can now use the `Project.prepare_audio()` method to prepare an `AudioDataset` that matches the properties of the `Transform` while containing `AudioData`s based on the `Event` list.

In [ ]:
ads = project.prepare_audio(
    transform=transform,
    events=unique_events,
)

In [ ]:
from pandas import DataFrame

# The audio data within ads have the properties of the transform (e.g. its sample rate)
# Their time bounds are set by the events list:
DataFrame(
    {"Begin": ad.begin, "End": ad.end, "Sample rate": ad.sample_rate} for ad in ads.data
)

Running the transform will compute the spectrograms of each audio data of the passed `audio_dataset` parameter and save the output files to disk.

If we don't want to edit the `SpectroDataset` that correspond to the exported spectrograms, we can directly run the transform (otherwise, refer to the example notebook on exporting spectrograms with the public API for more detail):

In [ ]:
project.run(
    transform=transform,
    audio_dataset=ads,  # We provide the custom AudioDataset that matches the events
)

In [ ]:
# Reset the project to get all files back to place.
project.reset()